# DATA 255 Lab 2: Super Resolution with TriScaleSR
Restores degraded 256x256x3 LR images to 256x256x3 HR images. Score is mean PSNR.

## 1. Data
Uses `train` and `valid` folders next to this notebook if present, otherwise copies them from Google Drive (Colab).

In [1]:
import os, shutil
ROOT = '.'
if not os.path.isdir('train'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    ROOT = '/content/data'
    for s in ['train', 'valid']:
        shutil.copytree(f'/content/drive/MyDrive/DATA255_Lab/{s}', f'{ROOT}/{s}', dirs_exist_ok=True)
for d, _, f in sorted(os.walk(ROOT)):
    if f and ('HR' in d or 'LR' in d):
        print(d, len(f))

./train/train/HR2 2217
./train/train/LR2 2217
./valid/HR2 110
./valid/LR2 110


## 2. Model
Three scale encoder decoder built from residual blocks (full, 1/2, 1/4 resolution) with a global skip, so the network only learns the correction to the LR image.
Only Conv, ConvTranspose, ReLU and Add are used so it compiles to the NPU. Input and output are 256x256x3.

In [2]:
%%writefile model.py
import torch.nn as nn

class RB(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(c, c, 3, 1, 1), nn.ReLU(True), nn.Conv2d(c, c, 3, 1, 1))

    def forward(self, x):
        return x + self.b(x)

class TriScaleSR(nn.Module):
    def __init__(self, c=64):
        super().__init__()
        self.head = nn.Conv2d(3, c, 3, 1, 1)
        self.e1 = nn.Sequential(RB(c), RB(c))
        self.d1 = nn.Sequential(nn.Conv2d(c, 2 * c, 3, 2, 1), nn.ReLU(True))
        self.e2 = nn.Sequential(RB(2 * c), RB(2 * c))
        self.d2 = nn.Sequential(nn.Conv2d(2 * c, 4 * c, 3, 2, 1), nn.ReLU(True))
        self.mid = nn.Sequential(*[RB(4 * c) for _ in range(6)])
        self.u2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, 2)
        self.f2 = nn.Sequential(RB(2 * c), RB(2 * c))
        self.u1 = nn.ConvTranspose2d(2 * c, c, 2, 2)
        self.f1 = nn.Sequential(RB(c), RB(c))
        self.tail = nn.Conv2d(c, 3, 3, 1, 1)
        nn.init.zeros_(self.tail.weight)
        nn.init.zeros_(self.tail.bias)

    def forward(self, x):
        f = self.head(x)
        a = self.e1(f)
        b = self.e2(self.d1(a))
        y = self.f2(b + self.u2(self.mid(self.d2(b))))
        y = self.f1(a + self.u1(y))
        return x + self.tail(y + f)

Overwriting model.py


## 3. Dataset and loaders
Images are kept in memory. Training uses random 192x192 crops with rotation and flip augmentation; validation uses full images.

In [3]:
import glob, inspect, random, time, numpy as np, torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from model import TriScaleSR

def load(p):
    im = Image.open(p).convert('RGB')
    return np.asarray(im if im.size == (256, 256) else im.resize((256, 256), Image.BICUBIC))

def files(split, kind):
    return sorted(glob.glob(f'{ROOT}/{split}/**/{kind}*/*.*', recursive=True))

class SRData(Dataset):
    def __init__(self, split, crop=None):
        lr, hr = files(split, 'LR'), files(split, 'HR')
        assert len(lr) == len(hr) > 0, (split, len(lr), len(hr))
        self.lr, self.hr = [torch.from_numpy(np.stack([load(p) for p in f])).permute(0, 3, 1, 2) for f in (lr, hr)]
        self.crop = crop

    def __len__(self):
        return len(self.lr)

    def __getitem__(self, i):
        lr, hr = self.lr[i], self.hr[i]
        if self.crop:
            y, x = random.randint(0, 256 - self.crop), random.randint(0, 256 - self.crop)
            lr, hr = lr[:, y:y + self.crop, x:x + self.crop], hr[:, y:y + self.crop, x:x + self.crop]
            k = random.randint(0, 3)
            lr, hr = lr.rot90(k, (1, 2)), hr.rot90(k, (1, 2))
            if random.random() < 0.5:
                lr, hr = lr.flip(2), hr.flip(2)
        return lr.float() / 255, hr.float() / 255

dev = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
train, valid = SRData('train', crop=192), SRData('valid')
tl = DataLoader(train, 16, shuffle=True, num_workers=0, drop_last=True)
vl = DataLoader(valid, 8)
psnr = lambda a, b: (-10 * torch.log10(((a.clamp(0, 1) - b) ** 2).mean((1, 2, 3)))).tolist()
print(len(train), 'train,', len(valid), 'valid, device', dev, torch.cuda.get_device_name() if dev == 'cuda' else '')
print('LR vs HR baseline PSNR: %.3f' % np.mean(sum([psnr(l, h) for l, h in vl], [])))

2217 train, 110 valid, device cuda NVIDIA GeForce RTX 5090
LR vs HR baseline PSNR: 25.121


## 4. Training
Adam with cosine learning rate decay over `ITERS` steps, mixed precision on GPU.
L1 loss for the first 80% of steps, then MSE (which matches PSNR) for the last 20%.
An exponential moving average (EMA) of the weights is evaluated every 2000 steps and the best one is saved to `best_model.pth`.
Lower `ITERS` if the printed time per 2000 steps is too slow for your GPU.

In [4]:
ITERS = int(os.environ.get('ITERS', 80000))
torch.manual_seed(0)
model = TriScaleSR().to(dev)
ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
print(sum(p.numel() for p in model.parameters()), 'params,', ITERS, 'steps')
optimizer = torch.optim.Adam(model.parameters(), 2e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, ITERS, 1e-6)
amp = dev == 'cuda'
scaler = torch.amp.GradScaler(enabled=amp)

def evaluate(m):
    m.eval()
    with torch.no_grad():
        return np.mean(sum([psnr(m(l.to(dev)).float().cpu(), h) for l, h in vl], []))

step, best, t0 = 0, 0, time.time()
while step < ITERS:
    for lr, hr in tl:
        model.train()
        lr, hr = lr.to(dev), hr.to(dev)
        with torch.autocast(dev, enabled=amp):
            out = model(lr).float()
        loss = F.l1_loss(out, hr) if step < 0.8 * ITERS else F.mse_loss(out, hr)
        # Backward pass
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        sched.step()
        ema.update_parameters(model)
        step += 1
        if step % 2000 == 0 or step == ITERS:
            v = evaluate(ema.module)
            if v > best:
                best = v
                torch.save(ema.module.state_dict(), 'best_model.pth')
            print(f'step {step} loss {loss.item():.4f} val PSNR {v:.3f} best {best:.3f} time {(time.time() - t0) / 60:.0f} min')
        if step == ITERS:
            break

9093635 params, 80000 steps
step 2000 loss 0.0313 val PSNR 25.594 best 25.594 time 4 min
step 4000 loss 0.0303 val PSNR 26.244 best 26.244 time 8 min
step 6000 loss 0.0284 val PSNR 26.392 best 26.392 time 12 min
step 8000 loss 0.0318 val PSNR 26.457 best 26.457 time 15 min
step 10000 loss 0.0313 val PSNR 26.495 best 26.495 time 19 min
step 12000 loss 0.0283 val PSNR 26.526 best 26.526 time 23 min
step 14000 loss 0.0289 val PSNR 26.544 best 26.544 time 26 min
step 16000 loss 0.0311 val PSNR 26.558 best 26.558 time 30 min
step 18000 loss 0.0264 val PSNR 26.568 best 26.568 time 34 min
step 20000 loss 0.0323 val PSNR 26.578 best 26.578 time 38 min
step 22000 loss 0.0275 val PSNR 26.583 best 26.583 time 42 min
step 24000 loss 0.0320 val PSNR 26.585 best 26.585 time 45 min
step 26000 loss 0.0369 val PSNR 26.588 best 26.588 time 49 min
step 28000 loss 0.0318 val PSNR 26.590 best 26.590 time 54 min
step 30000 loss 0.0328 val PSNR 26.593 best 26.593 time 58 min
step 32000 loss 0.0280 val PSNR 2

## 5. Evaluate best model and export ONNX
Reloads the saved weights into a fresh model (confirms architecture and weights match), reports validation PSNR and exports `sr_model.onnx` with input 1x3x256x256.

In [6]:
%pip install -q onnx
net = TriScaleSR().to(dev)
net.load_state_dict(torch.load('best_model.pth', map_location=dev))
print('best model val PSNR (PyTorch): %.3f' % evaluate(net))
kw = {'dynamo': False} if 'dynamo' in inspect.signature(torch.onnx.export).parameters else {}
torch.onnx.export(net.cpu(), torch.rand(1, 3, 256, 256), 'sr_model.onnx', opset_version=13, input_names=['input'], output_names=['output'], **kw)
print('saved best_model.pth, model.py, sr_model.onnx')

Note: you may need to restart the kernel to use updated packages.
best model val PSNR (PyTorch): 26.595
saved best_model.pth, model.py, sr_model.onnx
